# Srtforge Google Drive Colab Benchmark (Drive-First, Colab-Fixed)

This notebook is tuned for Google Colab and runs the paper benchmark profile:

- FV4 / Mel-Band Roformer vocal separation
- Faster-Whisper `large-v3-turbo`
- `int8_float16` compute type
- Gemini disabled
- WER computed with `jiwer.process_words`, matching Hugging Face/Evaluate WER behavior

It mounts Google Drive, installs the runtime dependencies, downloads the FV4 release assets, copies media from Drive to local Colab storage for faster processing, writes generated subtitles, metrics, and plots back to Drive, and reports both normal WER and the paper-style named-entity-adjusted WER for Srtforge rows.

## 1. Select GPU runtime

In Colab, use `Runtime -> Change runtime type -> T4 GPU` or better. Then run the cell below.

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

Because Colab cannot access private GitHub repositories or private release assets without credentials, the default workflow is Drive-first.

Expected Drive layout:

```text
MyDrive/srtforge-benchmark/
  source/
    Srtforge.zip        # zip of the repo root containing pyproject.toml and srtforge/

  assets/
    voc_fv4.ckpt
    voc_gabox.yaml
    download_checks.json
    models-scores.json

  media/
    S01E22.mkv

  references/
    S01E22.truth.txt
```

The two small JSON assets prevent `audio-separator` from trying to download metadata during FV4 model loading, which is important in Colab notebooks.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Configure paths and benchmark matrix

Edit this cell if your Drive folder or filenames differ. The default matrix reproduces the main FV4 + Whisper `int8_float16` row and the raw Whisper precision comparison rows.

In [ ]:
from pathlib import Path
import time

# Drive folder containing source/assets/media/reference inputs and receiving outputs.
DRIVE_ROOT = Path("/content/drive/MyDrive/srtforge-benchmark")
MEDIA_DIR = DRIVE_ROOT / "media"
REFERENCE_DIR = DRIVE_ROOT / "references"
ASSET_DIR = DRIVE_ROOT / "assets"
SOURCE_DIR = DRIVE_ROOT / "source" / "Srtforge"
SOURCE_ZIP = DRIVE_ROOT / "source" / "Srtforge.zip"
OUTPUT_ROOT = DRIVE_ROOT / "colab_runs"

# Source mode. Use "drive" unless the GitHub repo is public or you provide GITHUB_TOKEN.
SOURCE_MODE = "drive"  # "drive" or "github"
REPO_URL = "https://github.com/Kunal926/Srtforge.git"
SRTFORGE_REF = "main"  # branch/tag to clone if SOURCE_MODE == "github"
GITHUB_TOKEN = ""  # optional; prefer setting this with getpass or Colab secrets instead of hardcoding

# FV4 assets. Drive is the default because private GitHub release assets return 404 in Colab.
ALLOW_GITHUB_ASSET_DOWNLOAD = False
RELEASE_BASE = "https://github.com/Kunal926/Srtforge/releases/download/v1.0.0"

# Run mode.
RUN_MODE = "single"  # "single" or "batch"
SINGLE_EPISODE = "S01E22"

# Copying to /content is faster and more stable than reading large MKV files through Drive FUSE.
COPY_MEDIA_TO_LOCAL = True

# Reference handling.
ASS_STYLES = {"Default"}
SDH_REFERENCE = False
ALLOW_UNTAGGED_ENGLISH = False

# Paper profile plus precision comparison rows.
# Add "srtforge_fv_whisper_float32" if you want to measure FV4+fp32 too; it was not one of the final paper rows.
VARIANTS_TO_RUN = [
    "srtforge_fv_whisper_int8_float16",
    "raw_whisper_int8_float16",
    "raw_whisper_float32",
]

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
DRIVE_RUN_DIR = OUTPUT_ROOT / RUN_ID
LOCAL_WORK_DIR = Path("/content/srtforge_colab_work") / RUN_ID

for path in (MEDIA_DIR, REFERENCE_DIR, ASSET_DIR, SOURCE_ZIP.parent, DRIVE_RUN_DIR, LOCAL_WORK_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Drive source folder:", SOURCE_DIR)
print("Drive source zip:", SOURCE_ZIP)
print("Drive assets:", ASSET_DIR)
print("Drive run dir:", DRIVE_RUN_DIR)
print("Local work dir:", LOCAL_WORK_DIR)
print("Variants:", VARIANTS_TO_RUN)

## 4. Install Srtforge runtime dependencies

This installs only the Python pipeline pieces needed for Colab, not the desktop/Tauri app. It also copies the Srtforge source from Drive by default, so it works even when GitHub is private.

If you use Drive mode, upload either `source/Srtforge.zip` or an unzipped `source/Srtforge/` folder before running this cell.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

REPO_DIR = Path("/content/Srtforge")
EXTRACT_DIR = Path("/content/srtforge_source_extract")


def run(cmd, *, cwd=None, env=None):
    shown = []
    for part in cmd:
        text = str(part)
        if GITHUB_TOKEN and GITHUB_TOKEN in text:
            text = text.replace(GITHUB_TOKEN, "***")
        shown.append(text)
    print("$", " ".join(shown))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)


def find_repo_root(base: Path) -> Path:
    candidates = []
    for pyproject in base.rglob("pyproject.toml"):
        root = pyproject.parent
        if (root / "srtforge").is_dir():
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError(
            f"Could not find repo root under {base}. Expected pyproject.toml and srtforge/ package."
        )
    return sorted(candidates, key=lambda p: len(p.parts))[0]


def copy_source_from_drive():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    if SOURCE_DIR.exists() and (SOURCE_DIR / "pyproject.toml").exists():
        print("Copying source folder from Drive:", SOURCE_DIR)
        ignore = shutil.ignore_patterns(
            ".git", ".venv", "__pycache__", ".pytest_cache", "target", "node_modules", "dist", "build"
        )
        shutil.copytree(SOURCE_DIR, REPO_DIR, ignore=ignore)
        return
    if SOURCE_ZIP.exists():
        print("Extracting source zip from Drive:", SOURCE_ZIP)
        if EXTRACT_DIR.exists():
            shutil.rmtree(EXTRACT_DIR)
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(SOURCE_ZIP) as zf:
            zf.extractall(EXTRACT_DIR)
        root = find_repo_root(EXTRACT_DIR)
        ignore = shutil.ignore_patterns(
            ".git", ".venv", "__pycache__", ".pytest_cache", "target", "node_modules", "dist", "build"
        )
        shutil.copytree(root, REPO_DIR, ignore=ignore)
        return
    raise FileNotFoundError(
        "Missing Srtforge source. Upload one of these to Drive:\n"
        f"  {SOURCE_ZIP}\n"
        f"  {SOURCE_DIR}\n"
        "The zip/folder must contain pyproject.toml and the srtforge/ package."
    )


def clone_source_from_github():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    url = REPO_URL
    if GITHUB_TOKEN:
        url = url.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1)
    run(["git", "clone", "--depth", "1", "--branch", SRTFORGE_REF, url, str(REPO_DIR)])

# The r2u.stat.illinois.edu source warning in Colab is harmless; apt still installs ffmpeg.
run(["apt-get", "-qq", "update"])
run(["apt-get", "-qq", "install", "-y", "ffmpeg"])

if SOURCE_MODE.lower() == "drive":
    copy_source_from_drive()
elif SOURCE_MODE.lower() == "github":
    clone_source_from_github()
else:
    raise ValueError("SOURCE_MODE must be 'drive' or 'github'")

if not (REPO_DIR / "pyproject.toml").exists() or not (REPO_DIR / "srtforge").is_dir():
    raise FileNotFoundError(f"Invalid repo copy at {REPO_DIR}")

os.chdir(REPO_DIR)
print("Repo ready:", REPO_DIR)

run([sys.executable, "-m", "pip", "install", "-U", "-q", "pip", "wheel", "setuptools<82"])

# Keep this focused on the pipeline path used by the benchmark.
# PySide6, NeMo/Parakeet, and desktop packaging dependencies are intentionally not installed.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime", "onnxruntime-gpu"], check=False)
run([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy>=2.0.0,<3.0.0",
    "typer>=0.12.3",
    "rich>=13.7.1",
    "tqdm>=4.66.4",
    "pyyaml>=6.0.1",
    "requests>=2.32.3",
    "soundfile>=0.12.1",
    "matplotlib>=3.8.0",
    "pandas>=2.0.0",
    "jiwer>=3.0.0",
    "av>=14.0.0",
    "faster-whisper==1.2.1",
    "ctranslate2==4.7.1",
    "audio-separator==0.44.1",
    "onnxruntime-gpu[cuda,cudnn]>=1.23.0",
])
run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps", "-q"])

os.environ["SRTFORGE_PROJECT_ROOT"] = str(REPO_DIR)
os.environ["RICH_NO_COLOR"] = "1"
os.environ["TERM"] = "dumb"
os.environ["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")

print("Installed Srtforge from", REPO_DIR)

## 5. Runtime sanity check

This confirms that PyTorch, CTranslate2, and ONNX Runtime can see CUDA. FV4 separation depends on ONNX Runtime GPU; Faster-Whisper depends on CTranslate2 CUDA.

In [ ]:
import importlib
import os
import sys

import torch
import ctranslate2
import onnxruntime as ort

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("CTranslate2:", ctranslate2.__version__)
try:
    print("CTranslate2 CUDA device count:", ctranslate2.get_cuda_device_count())
except Exception as exc:
    print("CTranslate2 CUDA check failed:", repr(exc))

preload = getattr(ort, "preload_dlls", None)
if callable(preload):
    try:
        preload(cuda=True, cudnn=True, msvc=False, directory=None)
    except Exception as exc:
        print("ONNX preload warning:", repr(exc))

print("ONNX Runtime:", ort.__version__)
print("ONNX providers:", ort.get_available_providers())
if "CUDAExecutionProvider" not in ort.get_available_providers():
    print("WARNING: ONNX Runtime CUDAExecutionProvider is not visible. FV4 Roformer can still use Torch CUDA, but ONNX-only separator models would run on CPU.")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Switch Colab to a GPU runtime before running the benchmark.")

## 6. Prepare FV4 model assets

By default this cell copies `voc_fv4.ckpt` and `voc_gabox.yaml` from `MyDrive/srtforge-benchmark/assets/`. This avoids the 404 you saw when Colab tried to download private GitHub release assets.

Set `ALLOW_GITHUB_ASSET_DOWNLOAD = True` only if the release assets are public or you provide `GITHUB_TOKEN`.

In [ ]:
import hashlib
import importlib
import os
import shutil
from pathlib import Path

import requests

MODEL_DIR = Path("/content/srtforge_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
os.environ["AUDIO_SEPARATOR_MODEL_DIR"] = str(MODEL_DIR)

ASSETS = {
    "voc_fv4.ckpt": "1a9657de5fd3ed87ad4fd1a9d2069743ecb33424836973ad0f3288e2a64e90bc",
    "voc_gabox.yaml": "a4f9b0d143b5cb9d5d1d3d9414c50868107caadcac1faaf9e0f021f7bf3d1c8b",
    "download_checks.json": "d3622e1fa19c161d3cf704927711b453d593a3f1eb0f2e0838c3136907935151",
    "models-scores.json": "aa33a06cb8a583ee053a73bf47065690817fde6cd2c83c57b6f9f543da0dcc2a",
}


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fp:
        for chunk in iter(lambda: fp.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def copy_or_download_asset(name: str, expected_sha256: str) -> Path:
    # audio-separator expects download_checks.json in AUDIO_SEPARATOR_MODEL_DIR.
    # Its package metadata file models-scores.json is copied into the package below.
    target = MODEL_DIR / name
    drive_path = ASSET_DIR / name

    if target.exists() and sha256_file(target) == expected_sha256:
        print("OK:", target)
        return target

    if drive_path.exists():
        print("Copying asset from Drive:", drive_path)
        shutil.copy2(drive_path, target)
        actual = sha256_file(target)
        if actual != expected_sha256:
            raise RuntimeError(
                f"Bad SHA-256 for Drive asset {drive_path}: expected {expected_sha256}, got {actual}"
            )
        print("OK:", target)
        return target

    if ALLOW_GITHUB_ASSET_DOWNLOAD:
        url = f"{RELEASE_BASE}/{name}"
        headers = {}
        token = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN", "")
        if token:
            headers["Authorization"] = f"Bearer {token}"
        print("Downloading", url)
        with requests.get(url, headers=headers, stream=True, timeout=120) as response:
            if response.status_code == 404:
                raise FileNotFoundError(
                    f"GitHub returned 404 for {url}. If the repo/release is private, upload {name} to {ASSET_DIR} "
                    "or provide a token with release asset access."
                )
            response.raise_for_status()
            with target.open("wb") as fp:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        fp.write(chunk)
        actual = sha256_file(target)
        if actual != expected_sha256:
            raise RuntimeError(f"Bad SHA-256 for {name}: expected {expected_sha256}, got {actual}")
        print("OK:", target)
        return target

    raise FileNotFoundError(
        f"Missing {name}. Upload it to Drive at:\n  {drive_path}\n"
        "The direct GitHub release URL is not used unless ALLOW_GITHUB_ASSET_DOWNLOAD=True."
    )

for asset_name, digest in ASSETS.items():
    copy_or_download_asset(asset_name, digest)

# Force audio-separator to use the same offline model-score metadata as the packaged worker.
import audio_separator
pkg_dir = Path(audio_separator.__file__).resolve().parent
scores_src = MODEL_DIR / "models-scores.json"
scores_dst = pkg_dir / "models-scores.json"
if scores_src.exists():
    shutil.copy2(scores_src, scores_dst)
    print("Installed audio_separator models-scores.json:", scores_dst)

FV4_MODEL = MODEL_DIR / "voc_fv4.ckpt"
FV4_CONFIG = MODEL_DIR / "voc_gabox.yaml"
print("FV4 model:", FV4_MODEL)
print("FV4 config:", FV4_CONFIG)
print("Audio-separator model dir:", MODEL_DIR)

## 7. Discover Drive inputs

The notebook matches media and reference files by episode key, for example `S01E22.mkv` with `S01E22.truth.txt`. If your media filename is long, keep `S01E22` somewhere in the filename.

In [ ]:
import re
from dataclasses import dataclass
from pathlib import Path

EP_RE = re.compile(r"S\d{2}E\d{2}", re.IGNORECASE)
MEDIA_EXTS = {".mkv", ".mp4", ".mov", ".avi", ".webm", ".m4v"}
REF_EXTS = [".truth.txt", ".srt", ".ass", ".txt"]

@dataclass(frozen=True)
class BenchmarkItem:
    key: str
    media_path: Path
    reference_path: Path | None


def episode_key(path: Path) -> str:
    match = EP_RE.search(path.name)
    return match.group(0).upper() if match else path.stem


def find_reference(key: str) -> Path | None:
    candidates = []
    for ext in REF_EXTS:
        candidates.append(REFERENCE_DIR / f"{key}{ext}")
    candidates.extend(sorted(REFERENCE_DIR.glob(f"*{key}*")))
    for path in candidates:
        if path.exists() and path.is_file() and path.suffix.lower() in {".txt", ".srt", ".ass"}:
            return path
    return None


def discover_items() -> list[BenchmarkItem]:
    media_files = [p for p in sorted(MEDIA_DIR.iterdir()) if p.is_file() and p.suffix.lower() in MEDIA_EXTS]
    items = []
    for media in media_files:
        key = episode_key(media)
        if RUN_MODE == "single" and key != SINGLE_EPISODE.upper():
            continue
        items.append(BenchmarkItem(key=key, media_path=media, reference_path=find_reference(key)))
    return items

items = discover_items()
if not items:
    raise FileNotFoundError(f"No media files found in {MEDIA_DIR}. Expected files like S01E22.mkv")

for item in items:
    print(item.key)
    print("  media:", item.media_path)
    print("  ref:  ", item.reference_path or "none; WER will be skipped")

## 8. Text normalization and WER helpers

Normal WER uses the standard word-level formula. The named-entity-adjusted score is reported only for Srtforge/FV4 rows because the paper used it as a proxy for the disabled Gemini correction pass.

In [ ]:
import html
import json
import re
from pathlib import Path
from typing import Any

import jiwer

HTML_TAG_RE = re.compile(r"<[^>]+>")
ASS_TAG_RE = re.compile(r"\{[^}]*\}")
SRT_TIMESTAMP_RE = re.compile(r"\d{2}:\d{2}:\d{2},\d{3}\s+-->\s+\d{2}:\d{2}:\d{2},\d{3}")
BRACKET_RE = re.compile(r"\[[^\]]+\]|\([^)]*\)")
SPEAKER_PREFIX_RE = re.compile(r"^\s*(?:[A-Z][A-Z0-9 .'\-]{1,30}:)\s+")
PUNCT_RE = re.compile(r"[^a-z0-9\s']")
SPACE_RE = re.compile(r"\s+")

SINGLE_TOKEN_CANONICAL = {
    "'bout": ("about",), "'kay": ("okay",), "'less": ("unless",), "'em": ("them",),
    "ok": ("okay",), "gimme": ("give", "me"), "gonna": ("going", "to"), "gotta": ("got", "to"),
    "kinda": ("kind", "of"), "lemme": ("let", "me"), "sorta": ("sort", "of"), "wanna": ("want", "to"),
    "would've": ("would", "have"), "could've": ("could", "have"), "should've": ("should", "have"),
    "when're": ("when", "are"), "o'": ("of",), "y": ("you",), "rintarou": ("rintaro",),
    "kyouma": ("kyoma",), "cern": ("sern",), "sern": ("sern",),
}
MULTI_TOKEN_CANONICAL = {
    ("boogey", "man"): ("boogeyman",), ("nay", "sayers"): ("naysayers",),
    ("pay", "dirt"): ("paydirt",), ("un", "scoured"): ("unscoured",),
    ("meow", "ster"): ("meowster",), ("meow", "sters"): ("meowsters",),
}
NUMBER_WORDS = {
    "0": "zero", "1": "one", "2": "two", "3": "three", "4": "four", "5": "five", "6": "six",
    "7": "seven", "8": "eight", "9": "nine", "10": "ten", "11": "eleven", "12": "twelve",
    "13": "thirteen", "14": "fourteen", "15": "fifteen", "16": "sixteen", "17": "seventeen",
    "18": "eighteen", "19": "nineteen", "20": "twenty", "30": "thirty", "40": "forty",
    "50": "fifty", "60": "sixty", "70": "seventy", "80": "eighty", "90": "ninety",
}
ENTITY_SINGLE_ALIASES = {
    "akihabara", "amayuri", "amane", "aoyama", "braun", "cern", "christina", "christine",
    "comima", "dakihabara", "daru", "faris", "fb", "ferris", "feyris", "guiana", "hashida",
    "hououin", "ibn", "itaru", "john", "kiryuu", "kiriomoeka", "kiryu", "kiyama", "kurisu",
    "kuristina", "kyobe", "kyoma", "kyouma", "kyriomoica", "luka", "makise", "maki", "mayuri",
    "mayushii", "meowushii", "moeka", "nae", "nakabachi", "nishi", "ocarin", "okabe", "okarin",
    "okurisu", "oopa", "phonewave", "rintaro", "rintarou", "rukako", "ruka", "rumiho",
    "sekurisu", "sern", "shiina", "shinodaru", "suzuha", "teeter", "tennoji", "titor", "upa",
    "urishibara", "urushibara", "yanabayashi",
}
ENTITY_MULTI_ALIASES = {
    ("braun", "tube"), ("d", "mail"), ("divergence", "meter"), ("future", "gadget", "lab"),
    ("hoi", "and", "kilma"), ("hououin", "kyobe"), ("hououin", "kyoma"), ("hououin", "kyouma"),
    ("ibn", "5100"), ("ibn", "fifty", "one", "hundred"), ("john", "teeter"), ("john", "titor"),
    ("maki", "sekurisu"), ("makise", "kurisu"), ("may", "queen"), ("nishi", "azabu"),
    ("phone", "wave"), ("queen", "may's"), ("queen", "may"), ("radio", "building"),
    ("reading", "steiner"), ("shiina", "mayuri"), ("time", "leap"), ("urushibara", "ruka"),
    ("yanabayashi", "shrine"),
}


def parse_srt_text(path: Path) -> str:
    lines = []
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.isdigit() or SRT_TIMESTAMP_RE.search(line):
            continue
        lines.append(line)
    return " ".join(lines)


def parse_ass_text(path: Path, styles: set[str]) -> str:
    fields = None
    texts = []
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if line.lower().startswith("format:"):
            fields = [part.strip().lower() for part in line.split(":", 1)[1].split(",")]
            continue
        if not line.lower().startswith("dialogue:"):
            continue
        payload = line.split(":", 1)[1].strip()
        if fields:
            parts = payload.split(",", len(fields) - 1)
            data = dict(zip(fields, parts))
            style = data.get("style", "").strip()
            if styles and style not in styles:
                continue
            text = data.get("text", "")
        else:
            parts = payload.split(",", 9)
            text = parts[-1] if parts else payload
        texts.append(text)
    return " ".join(texts)


def load_reference_text(path: Path) -> str:
    suffixes = "".join(path.suffixes).lower()
    if suffixes.endswith(".srt"):
        return parse_srt_text(path)
    if suffixes.endswith(".ass"):
        return parse_ass_text(path, ASS_STYLES)
    return path.read_text(encoding="utf-8", errors="ignore")


def normalize_for_wer(text: str, *, sdh: bool = False) -> str:
    text = html.unescape(text)
    text = ASS_TAG_RE.sub("", text)
    text = HTML_TAG_RE.sub("", text)
    text = text.replace("\\N", " ").replace("\\n", " ").replace("\\h", " ")
    if sdh:
        text = BRACKET_RE.sub(" ", text)
        text = " ".join(SPEAKER_PREFIX_RE.sub("", line) for line in text.splitlines())
    text = text.lower()
    text = PUNCT_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text


def canonicalize_tokens(items: list[str]) -> list[str]:
    out = []
    idx = 0
    multi_lengths = sorted({len(key) for key in MULTI_TOKEN_CANONICAL}, reverse=True)
    while idx < len(items):
        matched = False
        for length in multi_lengths:
            phrase = tuple(items[idx:idx + length])
            replacement = MULTI_TOKEN_CANONICAL.get(phrase)
            if replacement is not None:
                out.extend(replacement)
                idx += length
                matched = True
                break
        if matched:
            continue
        word = items[idx]
        if word in SINGLE_TOKEN_CANONICAL:
            out.extend(SINGLE_TOKEN_CANONICAL[word])
        elif word in NUMBER_WORDS:
            out.append(NUMBER_WORDS[word])
        elif word.endswith("in'") and len(word) > 4:
            out.append(word[:-3] + "ing")
        else:
            out.append(word)
        idx += 1
    return out


def tokens_for_scoring(text: str) -> list[str]:
    normalized = normalize_for_wer(text, sdh=SDH_REFERENCE)
    return canonicalize_tokens(normalized.split()) if normalized else []


def entity_key(word: str) -> str:
    return word[:-2] if word.endswith("'s") else word


def contains_named_entity(items: list[str]) -> bool:
    keyed = [entity_key(word) for word in items]
    multi_lengths = sorted({len(key) for key in ENTITY_MULTI_ALIASES}, reverse=True)
    for idx, word in enumerate(keyed):
        if word in ENTITY_SINGLE_ALIASES:
            return True
        for length in multi_lengths:
            if tuple(keyed[idx:idx + length]) in ENTITY_MULTI_ALIASES:
                return True
    return False


def chunk_error_counts(kind: str, reference_count: int, hypothesis_count: int) -> tuple[int, int, int]:
    if kind == "substitute":
        return min(reference_count, hypothesis_count), max(0, reference_count - hypothesis_count), max(0, hypothesis_count - reference_count)
    if kind == "delete":
        return 0, reference_count, 0
    if kind == "insert":
        return 0, 0, hypothesis_count
    return 0, 0, 0


def word_metrics(reference: list[str], hypothesis: list[str]) -> dict[str, Any]:
    output = jiwer.process_words(" ".join(reference), " ".join(hypothesis))
    errors = int(output.substitutions + output.deletions + output.insertions)
    return {
        "wer": float(output.wer),
        "wer_pct": round(float(output.wer) * 100.0, 2),
        "hits": int(output.hits),
        "substitutions": int(output.substitutions),
        "deletions": int(output.deletions),
        "insertions": int(output.insertions),
        "reference_words": len(reference),
        "hypothesis_words": len(hypothesis),
        "word_errors": errors,
    }


def word_metrics_excluding_named_entity_errors(reference: list[str], hypothesis: list[str]) -> dict[str, Any]:
    output = jiwer.process_words(" ".join(reference), " ".join(hypothesis))
    excluded_substitutions = excluded_deletions = excluded_insertions = 0
    excluded_chunks = 0
    alignments = output.alignments[0] if output.alignments else []
    for chunk in alignments:
        if chunk.type == "equal":
            continue
        ref_span = reference[chunk.ref_start_idx:chunk.ref_end_idx]
        hyp_span = hypothesis[chunk.hyp_start_idx:chunk.hyp_end_idx]
        if not contains_named_entity(ref_span) and not contains_named_entity(hyp_span):
            continue
        substitutions, deletions, insertions = chunk_error_counts(chunk.type, len(ref_span), len(hyp_span))
        excluded_substitutions += substitutions
        excluded_deletions += deletions
        excluded_insertions += insertions
        excluded_chunks += 1
    substitutions = int(output.substitutions) - excluded_substitutions
    deletions = int(output.deletions) - excluded_deletions
    insertions = int(output.insertions) - excluded_insertions
    errors = substitutions + deletions + insertions
    wer = errors / len(reference) if reference else 0.0
    return {
        "wer": float(wer),
        "wer_pct": round(float(wer) * 100.0, 2),
        "hits": int(output.hits),
        "substitutions": substitutions,
        "deletions": deletions,
        "insertions": insertions,
        "reference_words": len(reference),
        "hypothesis_words": len(hypothesis),
        "word_errors": errors,
        "excluded_name_errors": excluded_substitutions + excluded_deletions + excluded_insertions,
        "excluded_name_chunks": excluded_chunks,
    }

## 9. Run benchmark variants

Each row writes a generated `.srt`, word timestamps, run summary, and metrics into the Drive run folder.

In [ ]:
import csv
import contextlib
import logging
import sys
import json
import os
import shutil
import subprocess
import time
from pathlib import Path
from typing import Any

from rich.console import Console
from srtforge.pipeline import PipelineConfig, run_pipeline
import srtforge.logging as srt_logging
import srtforge.pipeline as srt_pipeline
import srtforge.ffmpeg as srt_ffmpeg

# Rich live/status rendering can recurse inside Colab notebooks. Keep logs plain.
sys.setrecursionlimit(10000)
plain_console = Console(force_terminal=False, no_color=True, color_system=None, width=140, highlight=False)
srt_logging._console = plain_console
srt_ffmpeg.get_console = lambda: plain_console

@contextlib.contextmanager
def quiet_status(_message: str):
    yield

srt_logging.status = quiet_status
srt_pipeline.status = quiet_status
logging.getLogger("audio_separator").setLevel(logging.WARNING)

VARIANT_CONFIGS = {
    "srtforge_fv_whisper_int8_float16": {
        "separation_backend": "fv4",
        "whisper_compute_type": "int8_float16",
        "paper_adjust_names": True,
    },
    "srtforge_fv_whisper_float32": {
        "separation_backend": "fv4",
        "whisper_compute_type": "float32",
        "paper_adjust_names": True,
    },
    "srtforge_fv_whisper_float16": {
        "separation_backend": "fv4",
        "whisper_compute_type": "float16",
        "paper_adjust_names": True,
    },
    "raw_whisper_int8_float16": {
        "separation_backend": "none",
        "whisper_compute_type": "int8_float16",
        "paper_adjust_names": False,
    },
    "raw_whisper_float32": {
        "separation_backend": "none",
        "whisper_compute_type": "float32",
        "paper_adjust_names": False,
    },
    "raw_whisper_float16": {
        "separation_backend": "none",
        "whisper_compute_type": "float16",
        "paper_adjust_names": False,
    },
}

missing = [name for name in VARIANTS_TO_RUN if name not in VARIANT_CONFIGS]
if missing:
    raise ValueError(f"Unknown variants: {missing}. Known: {sorted(VARIANT_CONFIGS)}")


def ffprobe_duration(path: Path) -> float | None:
    cmd = [
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", str(path),
    ]
    try:
        out = subprocess.check_output(cmd, text=True).strip()
        return float(out) if out else None
    except Exception:
        return None


def copy_to_local(item: BenchmarkItem) -> tuple[Path, Path | None]:
    local_input_dir = LOCAL_WORK_DIR / "inputs" / item.key
    local_input_dir.mkdir(parents=True, exist_ok=True)
    if COPY_MEDIA_TO_LOCAL:
        local_media = local_input_dir / item.media_path.name
        if not local_media.exists() or local_media.stat().st_size != item.media_path.stat().st_size:
            print(f"Copying media to local Colab disk: {item.media_path.name}")
            shutil.copy2(item.media_path, local_media)
    else:
        local_media = item.media_path
    local_ref = None
    if item.reference_path:
        local_ref = local_input_dir / item.reference_path.name
        if not local_ref.exists() or local_ref.stat().st_size != item.reference_path.stat().st_size:
            shutil.copy2(item.reference_path, local_ref)
    return local_media, local_ref


def run_one(item: BenchmarkItem, variant_name: str) -> dict[str, Any]:
    variant = VARIANT_CONFIGS[variant_name]
    local_media, local_ref = copy_to_local(item)
    local_variant_dir = LOCAL_WORK_DIR / "runs" / item.key / variant_name
    drive_variant_dir = DRIVE_RUN_DIR / "runs" / item.key / variant_name
    local_variant_dir.mkdir(parents=True, exist_ok=True)
    drive_variant_dir.mkdir(parents=True, exist_ok=True)

    out_srt = local_variant_dir / f"{item.key}.{variant_name}.srt"
    word_ts = local_variant_dir / "word_timestamps.json"
    duration_s = ffprobe_duration(local_media)

    config = PipelineConfig(
        media_path=local_media,
        output_path=out_srt,
        models_dir=MODEL_DIR,
        fv4_model=FV4_MODEL,
        fv4_config=FV4_CONFIG,
        temp_dir=local_variant_dir / "tmp",
        output_directory=local_variant_dir,
        sample_rate=44100,
        separation_backend=variant["separation_backend"],
        separation_prefer_center=True,
        separation_prefer_gpu=True,
        ffmpeg_extraction_mode="dual_mono_center",
        ffmpeg_filter_chain="highpass=f=60,lowpass=f=10000,aformat=sample_fmts=flt,aresample=resampler=soxr:osf=flt:osr=16000",
        prefer_gpu=True,
        asr_engine="whisper",
        whisper_model="large-v3-turbo",
        whisper_language="en",
        whisper_compute_type=variant["whisper_compute_type"],
        gemini_enabled=False,
        allow_untagged_english=ALLOW_UNTAGGED_ENGLISH,
        dump_word_timestamps=True,
        word_timestamps_path=word_ts,
    )

    started = time.perf_counter()
    result = run_pipeline(config)
    wall_seconds = time.perf_counter() - started

    generated_srt = result.output_path or out_srt
    if result.failed or not generated_srt.exists():
        raise RuntimeError(f"Pipeline failed or produced no SRT at {generated_srt}. reason={result.reason!r}")

    row: dict[str, Any] = {
        "episode": item.key,
        "variant": variant_name,
        "engine": "whisper",
        "model": "large-v3-turbo",
        "separation_backend": variant["separation_backend"],
        "whisper_compute_type": variant["whisper_compute_type"],
        "gemini_enabled": False,
        "media_drive_path": str(item.media_path),
        "reference_drive_path": str(item.reference_path) if item.reference_path else "",
        "duration_s": duration_s,
        "wall_seconds": round(wall_seconds, 3),
        "rtf": round(wall_seconds / duration_s, 5) if duration_s else None,
        "srt_path": str(drive_variant_dir / generated_srt.name),
        "status": "completed",
    }

    if local_ref:
        reference_text = load_reference_text(local_ref)
        hypothesis_text = parse_srt_text(generated_srt)
        ref_tokens = tokens_for_scoring(reference_text)
        hyp_tokens = tokens_for_scoring(hypothesis_text)
        normal = word_metrics(ref_tokens, hyp_tokens)
        row.update({f"normal_{k}": v for k, v in normal.items()})
        row["wer_pct"] = normal["wer_pct"]
        row["reference_words"] = normal["reference_words"]
        row["word_errors"] = normal["word_errors"]
        if variant["paper_adjust_names"]:
            adjusted = word_metrics_excluding_named_entity_errors(ref_tokens, hyp_tokens)
            row.update({f"name_adjusted_{k}": v for k, v in adjusted.items()})
            row["paper_wer_pct"] = adjusted["wer_pct"]
            row["paper_word_errors"] = adjusted["word_errors"]
        else:
            row["paper_wer_pct"] = normal["wer_pct"]
            row["paper_word_errors"] = normal["word_errors"]

    summary_path = local_variant_dir / "run_summary.json"
    summary_path.write_text(json.dumps(row, indent=2), encoding="utf-8")

    # Persist important artifacts to Drive.
    for path in local_variant_dir.glob("*"):
        if path.is_file():
            shutil.copy2(path, drive_variant_dir / path.name)

    print(json.dumps({k: row.get(k) for k in ["episode", "variant", "wer_pct", "paper_wer_pct", "rtf", "wall_seconds"]}, indent=2))
    return row

rows = []
for item in items:
    for variant_name in VARIANTS_TO_RUN:
        print(f"\n=== {item.key} / {variant_name} ===")
        try:
            rows.append(run_one(item, variant_name))
        except Exception as exc:
            error_row = {
                "episode": item.key,
                "variant": variant_name,
                "status": "failed",
                "error": repr(exc),
            }
            rows.append(error_row)
            print("FAILED:", repr(exc))
            raise

metrics_json = DRIVE_RUN_DIR / "metrics.json"
metrics_csv = DRIVE_RUN_DIR / "metrics.csv"
metrics_json.write_text(json.dumps(rows, indent=2), encoding="utf-8")

fieldnames = sorted({key for row in rows for key in row})
with metrics_csv.open("w", encoding="utf-8", newline="") as fp:
    writer = csv.DictWriter(fp, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("Wrote:", metrics_json)
print("Wrote:", metrics_csv)

## 10. Summarize and plot results

For Srtforge/FV4 rows, `paper_wer_pct` is the named-entity-adjusted WER used in the paper. For raw rows, `paper_wer_pct` equals normal WER.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

metrics_path = DRIVE_RUN_DIR / "metrics.csv"
df = pd.read_csv(metrics_path)
completed = df[df["status"] == "completed"].copy()

cols = [
    "episode", "variant", "separation_backend", "whisper_compute_type",
    "normal_wer_pct", "paper_wer_pct", "rtf", "wall_seconds", "reference_words", "word_errors",
]
print(completed[[c for c in cols if c in completed.columns]].to_string(index=False))

fig_dir = DRIVE_RUN_DIR / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

if "paper_wer_pct" in completed.columns and completed["paper_wer_pct"].notna().any():
    plot_df = completed.sort_values(["episode", "paper_wer_pct"])
    labels = plot_df["episode"].astype(str) + "\n" + plot_df["variant"].astype(str).str.replace("srtforge_fv_whisper_", "FV4 ").str.replace("raw_whisper_", "raw ")
    colors = ["#2563eb" if str(v).startswith("srtforge") else "#dc2626" for v in plot_df["variant"]]
    plt.figure(figsize=(max(8, len(plot_df) * 1.2), 4.8))
    bars = plt.bar(range(len(plot_df)), plot_df["paper_wer_pct"], color=colors, edgecolor="#222", linewidth=0.8)
    plt.ylabel("WER (%)")
    plt.title("Srtforge Colab Benchmark WER")
    plt.xticks(range(len(plot_df)), labels, rotation=20, ha="right")
    plt.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, plot_df["paper_wer_pct"]):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05, f"{val:.2f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    out = fig_dir / "wer_by_variant.png"
    plt.savefig(out, dpi=180)
    plt.show()
    print("Saved", out)

if "rtf" in completed.columns and completed["rtf"].notna().any():
    plot_df = completed.sort_values(["episode", "rtf"])
    labels = plot_df["episode"].astype(str) + "\n" + plot_df["variant"].astype(str).str.replace("srtforge_fv_whisper_", "FV4 ").str.replace("raw_whisper_", "raw ")
    colors = ["#2563eb" if str(v).startswith("srtforge") else "#dc2626" for v in plot_df["variant"]]
    plt.figure(figsize=(max(8, len(plot_df) * 1.2), 4.8))
    bars = plt.bar(range(len(plot_df)), plot_df["rtf"], color=colors, edgecolor="#222", linewidth=0.8)
    plt.ylabel("Real-time factor")
    plt.title("Srtforge Colab Benchmark Runtime")
    plt.xticks(range(len(plot_df)), labels, rotation=20, ha="right")
    plt.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, plot_df["rtf"]):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003, f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    out = fig_dir / "rtf_by_variant.png"
    plt.savefig(out, dpi=180)
    plt.show()
    print("Saved", out)

## 11. Where outputs are saved

Open the printed Drive folder to download generated SRTs, JSON summaries, CSV metrics, and plots.

In [ ]:
print("Drive run folder:", DRIVE_RUN_DIR)
print("Metrics CSV:", DRIVE_RUN_DIR / "metrics.csv")
print("Metrics JSON:", DRIVE_RUN_DIR / "metrics.json")
print("Figures:", DRIVE_RUN_DIR / "figures")